# Lab06.2 – AXI Register Access

**Objective:** access the existing LED AXI GPIO through low-level MMIO. This is **not yet hardware acceleration**: it introduces PS–PL register communication.

**Safety:** restores LEDs to OFF at the end.

**Step 1 – Prepare.** Import PYNQ, load the base overlay and locate the LED GPIO register base address from metadata, avoiding hardcoded physical addresses.

In [ ]:
from pynq import Overlay, MMIO
import time
base = Overlay('base.bit')
led = base.ip_dict['leds_gpio']
print('LED controller: 0x{:08X}'.format(led['phys_addr']))
mmio = MMIO(led['phys_addr'], led.get('addr_range', 0x1000))
DATA = 0x00
TRI = 0x04 # bit=0 output, bit=1 input

**Step 2 – Configure the LED register.** Write `0` to the direction register, declaring the lowest four bits as outputs. Read back the direction register to verify.

In [ ]:
mmio.write(TRI, 0x00000000)
print('TRI:', hex(mmio.read(TRI)))

**Step 3 – Observe a sequence.** Light LD0, LD1, LD2 and LD3 one at a time. The finally block guarantees that LEDs are switched off even if interrupted.

In [ ]:
try:
    for i in range(4):
        value = 1 << i
        mmio.write(DATA, value)
        print('LD{} ON, DATA=0x{:X}'.format(i, mmio.read(DATA)))
        time.sleep(0.4)
finally:
    mmio.write(DATA, 0)
print('All LEDs OFF')

**Step 4 – Register readback.** Verify the current output register and state the difference between writing a register and executing a computation in PL.

In [ ]:
print('LED DATA register:', hex(mmio.read(DATA)))
assert (mmio.read(DATA) & 0xF) == 0

## Student questions
1. Explain the roles of AXI GPIO DATA and TRI.
2. Why should peripheral base addresses be obtained from `ip_dict` rather than copied from another overlay?
3. What is executed in PL when the ARM writes `0b1010` to the LED register?